# SHIPP Stage 9D — Semantic score per (request, listing) for Gold scoring v2

For every open Request: search its cleaned `request_text` inside its own category → `shipp_silver.silver_semantic_scores`.

**Grain:** one row per (`request_id`, `listing_id`) retrieved in the top `SEMANTIC_TOP_K_PER_REQUEST`.
`semantic_score` = search score clamped to [0, 1]. Pairs not retrieved are absent; Gold v2 treats absent as 0.
This notebook does **not** change Gold — see `rag/README.md` → "Switching Gold to v2".

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `silver_semantic_scores`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

In [0]:
from config.settings import SEMANTIC_TOP_K_PER_REQUEST
from rag.retrieval import search_listing_context
from rag.semantic import semantic_rows

TARGET_TABLE = SILVER_SEMANTIC_SCORES
EXPECTED_COLUMNS = ["request_id", "listing_id", "semantic_score", "retrieval_rank", "index_name", "scored_at"]
k = min(SEMANTIC_TOP_K_PER_REQUEST, SEARCH_MAX_K)

open_requests = (spark.table(SILVER_REQUESTS)
                 .filter(F.col("status").isin(ELIGIBLE_REQUEST_STATUSES))
                 .select("request_id", "request_text", "category").collect())
rows, skipped = [], []
for r in open_requests:
    if not r["request_text"]:
        skipped.append(r["request_id"])
        continue
    rows += semantic_rows(r["request_id"], search_listing_context(r["request_text"], k=k,
                                                                  category=r["category"], workspace_client=w))
print(f"Open requests: {len(open_requests)} | scored pairs: {len(rows)} | skipped (no text): {skipped}")

scores = (spark.createDataFrame(rows, "request_id string, listing_id string, semantic_score double, retrieval_rank int")
          .withColumn("index_name", F.lit(VS_INDEX_NAME))
          .withColumn("scored_at", F.current_timestamp())
          .select(*EXPECTED_COLUMNS))
row_count = scores.count()
display(scores.orderBy("request_id", "retrieval_rank"))

In [0]:
gold_ids = spark.table(GOLD_LISTING_SEARCH_DOCS).select("listing_id")
run_quality_gate({
    "duplicate (request, listing)": duplicate_key_count(scores, "request_id", "listing_id"),
    "score outside [0,1]": scores.filter(~F.col("semantic_score").between(0, 1)).count(),
    "listing not in search docs": scores.join(gold_ids, "listing_id", "left_anti").count(),
}, row_count=row_count, allow_empty=True)

written = write_overwrite(scores, TARGET_TABLE)
assert written == row_count, f"WRITE VALIDATION FAILED: wrote {written}, expected {row_count}"
assert_exact_schema(TARGET_TABLE, EXPECTED_COLUMNS)
summary = {"table": TARGET_TABLE, "rows": written, "requests": len(open_requests), "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))